In [3]:
"""
=============================================================================
  EGESZSEGBIZTOSITASI KOLTSEGEK - OLS LINEARIS REGRESSZIOS ELEMZES
  (Medical Insurance Cost - OLS Linear Regression Analysis)
=============================================================================

  Szerzo / Author : [Neved ide / Your Name Here]
  Datum  / Date   : 2026-10-16
  Python          : 3.10+
  Adatforras      : Kaggle - mragpavank/insurance1

  Celvaltozo (y)  : charges  (biztositasi koltseg)
  Magyarazo valtozok (X): age, sex, bmi, children, smoker, region

  A kod Jupyter Notebook strukturaju: minden szekciot "# %%"-jel valaszt el,
  igy barmely IDE-ben (VS Code, PyCharm, Spyder) cellankent futtathatonak kell
  lennie.
=============================================================================
"""

'\n=============================================================================\n  EGESZSEGBIZTOSITASI KOLTSEGEK - OLS LINEARIS REGRESSZIOS ELEMZES\n  (Medical Insurance Cost - OLS Linear Regression Analysis)\n=============================================================================\n\n  Szerzo / Author : [Neved ide / Your Name Here]\n  Datum  / Date   : 2026-10-16\n  Python          : 3.10+\n  Adatforras      : Kaggle - mragpavank/insurance1\n\n  Celvaltozo (y)  : charges  (biztositasi koltseg)\n  Magyarazo valtozok (X): age, sex, bmi, children, smoker, region\n\n  A kod Jupyter Notebook strukturaju: minden szekciot "# %%"-jel valaszt el,\n  igy barmely IDE-ben (VS Code, PyCharm, Spyder) cellankent futtathatonak kell\n  lennie.\n=============================================================================\n'

In [4]:
# 0. KONYVTARAK IMPORTALASA (Library Imports)
# ============================================================================
#import sys
#sys.stdout.reconfigure(encoding="utf-8", errors="replace")

import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd

# --- Vizualizacio (Agg backend: nem-interaktiv, nem blokkol) ---
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

# --- Statisztikai modellezes (Statsmodels) ---
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

# --- Normalitasvizsgalat ---
from scipy.stats import shapiro, jarque_bera

# --- Hibaertekek ---
from sklearn.metrics import mean_squared_error, mean_absolute_error

# --- Adatbetoltes Kaggle-rol ---
import kagglehub
from kagglehub import KaggleDatasetAdapter

# --- Kimeneti mappa az abraknak ---
OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath('')), "plots")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Globalis abra-szamlalo
_plot_counter = [0]


def save_and_show(fig, name_hint="plot"):
    """Abra mentese PNG-be es memoria felszabaditasa."""
    _plot_counter[0] += 1
    fname = f"{_plot_counter[0]:02d}_{name_hint}.png"
    fpath = os.path.join(OUTPUT_DIR, fname)
    fig.savefig(fpath, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"   [ABRA MENTVE] -> {fpath}")


# --- Abra stilus beallitas ---
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

print("[OK] Minden konyvtar sikeresen importalva.")

[OK] Minden konyvtar sikeresen importalva.


In [5]:
# 1. ADATBAZIS BEOLVASASA ES ALAPVETO ELLENORZES
#    (Data Loading & Basic Inspection)
# ============================================================================

# 1.0 - Adat beolvasas Kaggle-rol
df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "mragpavank/insurance1",
    "insurance.csv",
)

print("=" * 70)
print("1. ADATBAZIS BEOLVASASA ES ALAPVETO ELLENORZES")
print("=" * 70)

# 1.1 - Az elso es utolso 5 sor megjelenitese
print("\n--- Az adatbazis elso 5 sora ---")
print(df.head())

print("\n--- Az adatbazis utolso 5 sora ---")
print(df.tail())

# 1.2 - Adattipusok (dtypes)
print("\n--- Adattipusok (dtypes) ---")
print(df.dtypes)

# 1.3 - Altalanos informacio
print("\n--- Altalanos informacio (info) ---")
df.info()

# 1.4 - Duplikaumok vizsgalata
num_duplicates = df.duplicated().sum()
print(f"\n--- Duplikaumok szama: {num_duplicates} ---")
if num_duplicates > 0:
    print("[!] Duplikalt sorok talalhatok! Eltavolitas...")
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"   Uj adatbazis merete: {df.shape}")
else:
    print("[OK] Nincsenek duplikalt sorok.")

1. ADATBAZIS BEOLVASASA ES ALAPVETO ELLENORZES

--- Az adatbazis elso 5 sora ---
   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520

--- Az adatbazis utolso 5 sora ---
      age     sex    bmi  children smoker     region     charges
1333   50    male  30.97         3     no  northwest  10600.5483
1334   18  female  31.92         0     no  northeast   2205.9808
1335   18  female  36.85         0     no  southeast   1629.8335
1336   21  female  25.80         0     no  southwest   2007.9450
1337   61  female  29.07         0    yes  northwest  29141.3603

--- Adattipusok (dtypes) ---
age           int64
sex             str
bmi         float64
children      int64

In [6]:
# 2. LEIRO STATISZTIKAK ES FELTARO ADATELEMZES (EDA)
#    (Descriptive Statistics & Exploratory Data Analysis)
# ============================================================================

print("\n" + "=" * 70)
print("2. LEIRO STATISZTIKAK ES FELTARO ADATELEMZES (EDA)")
print("=" * 70)

# 2.1 - Megfigyelesek es valtozok szama
n_rows, n_cols = df.shape
print(f"\n[STAT] Megfigyelesek (sorok) szama : {n_rows}")
print(f"[STAT] Valtozok (oszlopok) szama   : {n_cols}")

# 2.2 - Leiro statisztikak (numerikus valtozokra)
print("\n--- Leiro statisztikak (numerikus valtozok) ---")
print(df.describe().round(2))

# 2.3 - Leiro statisztikak (kategorikus valtozokra)
print("\n--- Leiro statisztikak (kategorikus valtozok) ---")
print(df.describe(include="object"))

# 2.4 - Hianyzo (missing) ertekek vizsgalata
print("\n--- Hianyzo ertekek osszesitese ---")
missing_df = pd.DataFrame({
    "Hianyzo ertekek szama": df.isnull().sum(),
    "Hianyzo ertekek aranya (%)": (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing_df)

if df.isnull().sum().sum() > 0:
    print("\n[!] Hianyzo adatok talalhatok! Kezeles szukseges...")
    # Numerikus valtozok: median imputacio (robusztus kiugro ertekek eseten)
    num_cols = df.select_dtypes(include=np.number).columns
    for col in num_cols:
        if df[col].isnull().sum() > 0:
            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
            print(f"   -> '{col}' hianyzo ertekei mediannal ({median_val}) potolva.")
    # Kategorikus valtozok: modusz (leggyakoribb ertek) imputacio
    cat_cols = df.select_dtypes(include="object").columns
    for col in cat_cols:
        if df[col].isnull().sum() > 0:
            mode_val = df[col].mode()[0]
            df[col].fillna(mode_val, inplace=True)
            print(f"   -> '{col}' hianyzo ertekei modusszal ('{mode_val}') potolva.")
    print("[OK] Hianyzo ertekek kezelve.")
else:
    print("[OK] Nincsenek hianyzo ertekek az adatbazisban.")


2. LEIRO STATISZTIKAK ES FELTARO ADATELEMZES (EDA)

[STAT] Megfigyelesek (sorok) szama : 1337
[STAT] Valtozok (oszlopok) szama   : 7

--- Leiro statisztikak (numerikus valtozok) ---
           age      bmi  children   charges
count  1337.00  1337.00   1337.00   1337.00
mean     39.22    30.66      1.10  13279.12
std      14.04     6.10      1.21  12110.36
min      18.00    15.96      0.00   1121.87
25%      27.00    26.29      0.00   4746.34
50%      39.00    30.40      1.00   9386.16
75%      51.00    34.70      2.00  16657.72
max      64.00    53.13      5.00  63770.43

--- Leiro statisztikak (kategorikus valtozok) ---
         sex smoker     region
count   1337   1337       1337
unique     2      2          4
top     male     no  southeast
freq     675   1063        364

--- Hianyzo ertekek osszesitese ---
          Hianyzo ertekek szama  Hianyzo ertekek aranya (%)
age                           0                         0.0
sex                           0                         0.

In [7]:
# 2.5 - ELOSZLAS-ABRAK (Distribution Plots)
# ============================================================================

# Numerikus es kategorikus valtozok szetvalasztasa
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include="object").columns.tolist()

print(f"\n[NUM] Numerikus valtozok  : {numeric_cols}")
print(f"[CAT] Kategorikus valtozok: {categorical_cols}")

# --- 2.5a: Numerikus valtozok - Hisztogram + KDE ---
print("\n--- Numerikus valtozok eloszlasa (Hisztogram + KDE) ---")
for col in numeric_cols:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Hisztogram + KDE
    sns.histplot(df[col], kde=True, bins=30, color="steelblue", ax=axes[0],
                 edgecolor="white", linewidth=0.5)
    axes[0].set_title(f"'{col}' eloszlasa (Hisztogram + KDE)")
    axes[0].set_xlabel(col)
    axes[0].set_ylabel("Gyakorisag")

    # Boxplot (kiugro ertekek vizualizalasara)
    sns.boxplot(x=df[col], color="lightcoral", ax=axes[1],
                flierprops=dict(marker="o", markersize=4, alpha=0.5))
    axes[1].set_title(f"'{col}' boxplot (kiugro ertekek)")
    axes[1].set_xlabel(col)

    plt.tight_layout()
    save_and_show(fig, f"dist_{col}")

# --- 2.5b: Kategorikus valtozok - Countplot ---
print("\n--- Kategorikus valtozok eloszlasa (Countplot) ---")
for col in categorical_cols:
    fig, ax = plt.subplots(figsize=(8, 5))
    order = df[col].value_counts().index
    sns.countplot(x=df[col], order=order, palette="viridis", ax=ax,
                  edgecolor="black", linewidth=0.5)
    ax.set_title(f"'{col}' ertekeinek gyakorisaga (Countplot)")
    ax.set_xlabel(col)
    ax.set_ylabel("Gyakorisag")

    # Ertekek megjelenitese az oszlopok tetejen
    for p in ax.patches:
        ax.annotate(f"{int(p.get_height())}",
                    (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha="center", va="bottom", fontsize=10, fontweight="bold")
    plt.tight_layout()
    save_and_show(fig, f"count_{col}")


[NUM] Numerikus valtozok  : ['age', 'bmi', 'children', 'charges']
[CAT] Kategorikus valtozok: ['sex', 'smoker', 'region']

--- Numerikus valtozok eloszlasa (Hisztogram + KDE) ---
   [ABRA MENTVE] -> c:\Users\hp\Antigravity\plots\01_dist_age.png
   [ABRA MENTVE] -> c:\Users\hp\Antigravity\plots\02_dist_bmi.png
   [ABRA MENTVE] -> c:\Users\hp\Antigravity\plots\03_dist_children.png
   [ABRA MENTVE] -> c:\Users\hp\Antigravity\plots\04_dist_charges.png

--- Kategorikus valtozok eloszlasa (Countplot) ---
   [ABRA MENTVE] -> c:\Users\hp\Antigravity\plots\05_count_sex.png
   [ABRA MENTVE] -> c:\Users\hp\Antigravity\plots\06_count_smoker.png
   [ABRA MENTVE] -> c:\Users\hp\Antigravity\plots\07_count_region.png


In [8]:
# 3. ADATELOKESZITES ES SPEARMAN KORRELACIO VIZSGALAT
#    (Data Preparation & Spearman Correlation Analysis)
# ============================================================================

print("\n" + "=" * 70)
print("3. ADATELOKESZITES ES SPEARMAN KORRELACIO")
print("=" * 70)

# 3.1 - One-Hot Encoding a kategorikus valtozokra
#   drop_first=True -> a Dummy Variable Trap elkerulese erdekeben
#   dtype=float -> statsmodels kompatibilitas
print("\n--- One-Hot Encoding (pd.get_dummies) ---")
df_encoded = pd.get_dummies(df, columns=categorical_cols,
                            drop_first=True, dtype=float)
print(f"   Eredeti adatbazis merete : {df.shape}")
print(f"   Kodolt adatbazis merete  : {df_encoded.shape}")
print(f"\n   Uj oszlopok: {list(df_encoded.columns)}")
print(df_encoded.head())

# 3.2 - Spearman-fele rangkorrelacios matrix
print("\n--- Spearman-fele rangkorrelacios matrix ---")
spearman_corr = df_encoded.corr(method="spearman")
print(spearman_corr.round(2))

# 3.3 - Spearman korrelacio hoterkep (Heatmap)
fig_hm = plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(spearman_corr, dtype=bool))  # Felso haromszog maszkolasa
sns.heatmap(spearman_corr, annot=True, fmt=".2f", cmap="coolwarm",
            mask=mask, linewidths=0.5, linecolor="white",
            vmin=-1, vmax=1, center=0,
            cbar_kws={"shrink": 0.8, "label": "Spearman rho"})
plt.title("Spearman-fele Rangkorrelacio Hoterkep", fontsize=16, fontweight="bold")
plt.tight_layout()
save_and_show(fig_hm, "spearman_heatmap")


3. ADATELOKESZITES ES SPEARMAN KORRELACIO

--- One-Hot Encoding (pd.get_dummies) ---
   Eredeti adatbazis merete : (1337, 7)
   Kodolt adatbazis merete  : (1337, 9)

   Uj oszlopok: ['age', 'bmi', 'children', 'charges', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']
   age     bmi  children      charges  sex_male  smoker_yes  region_northwest  \
0   19  27.900         0  16884.92400       0.0         1.0               0.0   
1   18  33.770         1   1725.55230       1.0         0.0               0.0   
2   28  33.000         3   4449.46200       1.0         0.0               0.0   
3   33  22.705         0  21984.47061       1.0         0.0               1.0   
4   32  28.880         0   3866.85520       1.0         0.0               1.0   

   region_southeast  region_southwest  
0               0.0               1.0  
1               1.0               0.0  
2               1.0               0.0  
3               0.0               0.0  
4     

In [9]:
# 4. KORRELACIO-ALAPU SZURES ES MULTIKOLLINEARITAS (VIF)
#    (Correlation-Based Filtering & Variance Inflation Factor)
# ============================================================================

print("\n" + "=" * 70)
print("4. KORRELACIO-ALAPU SZURES ES VIF VIZSGALAT")
print("=" * 70)

# --- 4.0 - Celvaltozo es magyarazo valtozok definialasa ---
target = "charges"
X_cols = [c for c in df_encoded.columns if c != target]
y = df_encoded[target].copy()

print(f"\n   Celvaltozo (y)           : {target}")
print(f"   Magyarazo valtozok (X)   : {X_cols}")

# --- 4.1 - Magas korrelacio szurese a magyarazo valtozok kozott ---
print("\n--- 4.1 Magas korrelacio szurese (|r| > 0.7) ---")

corr_X = df_encoded[X_cols].corr(method="spearman")
dropped_corr = []  # Kiszurt valtozok listaja

# Vegigmegyunk az osszes valtozo-paron (felso haromszog)
threshold = 0.7

for i in range(len(X_cols)):
    for j in range(i + 1, len(X_cols)):
        var1, var2 = X_cols[i], X_cols[j]
        corr_val = abs(corr_X.loc[var1, var2])

        if corr_val > threshold:
            # Melyiknek alacsonyabb a korrelacija a celvaltozoval?
            corr_y_var1 = abs(spearman_corr.loc[var1, target])
            corr_y_var2 = abs(spearman_corr.loc[var2, target])

            if corr_y_var1 < corr_y_var2:
                to_drop = var1
            else:
                to_drop = var2

            if to_drop not in dropped_corr:
                dropped_corr.append(to_drop)
                print(f"   [!] |rho({var1}, {var2})| = {corr_val:.4f} > {threshold}")
                print(f"       -> '{to_drop}' kiesik (alacsonyabb korrelacio a celvaltozoval)")
                print(f"         rho({var1}, {target}) = {corr_y_var1:.4f}, "
                      f"rho({var2}, {target}) = {corr_y_var2:.4f}")

if not dropped_corr:
    print("   [OK] Nincs olyan valtozo-par, amelynel |r| > 0.7 -> nincs szures.")
else:
    X_cols = [c for c in X_cols if c not in dropped_corr]
    print(f"\n   Kiszurt valtozok : {dropped_corr}")
    print(f"   Marado valtozok  : {X_cols}")

# --- 4.2 - VIF (Variance Inflation Factor) vizsgalat ---
print("\n--- 4.2 VIF (Variance Inflation Factor) vizsgalat ---")


def calculate_vif(X_data):
    """VIF ertekek kiszamitasa konstanssal kiegeszitett X matrixra."""
    X_const = sm.add_constant(X_data)
    vif_data = pd.DataFrame()
    vif_data["Valtozo"] = X_const.columns
    vif_data["VIF"] = [variance_inflation_factor(X_const.values, i)
                       for i in range(X_const.shape[1])]
    return vif_data


# Iterativ VIF szures: amig van VIF > 5, a legmagasabbat kiszurjuk
vif_iteration = 0
X_vif = df_encoded[X_cols].copy()

while True:
    vif_df = calculate_vif(X_vif)
    # Konstans kiszurese a vizsgalatbol
    vif_no_const = vif_df[vif_df["Valtozo"] != "const"]
    max_vif = vif_no_const["VIF"].max()
    max_vif_var = vif_no_const.loc[vif_no_const["VIF"].idxmax(), "Valtozo"]

    vif_iteration += 1
    print(f"\n   --- VIF Iteracio #{vif_iteration} ---")
    print(vif_no_const.to_string(index=False))

    if max_vif > 5:
        print(f"\n   [!] '{max_vif_var}' VIF = {max_vif:.2f} > 5 -> Kiszures!")
        X_vif = X_vif.drop(columns=[max_vif_var])
    else:
        print(f"\n   [OK] Minden valtozo VIF < 5. Multikollinearitas nem problema.")
        break

X_cols_final = X_vif.columns.tolist()
print(f"\n   Vegso magyarazo valtozok a VIF szures utan: {X_cols_final}")


4. KORRELACIO-ALAPU SZURES ES VIF VIZSGALAT

   Celvaltozo (y)           : charges
   Magyarazo valtozok (X)   : ['age', 'bmi', 'children', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']

--- 4.1 Magas korrelacio szurese (|r| > 0.7) ---
   [OK] Nincs olyan valtozo-par, amelynel |r| > 0.7 -> nincs szures.

--- 4.2 VIF (Variance Inflation Factor) vizsgalat ---

   --- VIF Iteracio #1 ---
         Valtozo      VIF
             age 1.016794
             bmi 1.106742
        children 1.004017
        sex_male 1.008944
      smoker_yes 1.012100
region_northwest 1.517673
region_southeast 1.651779
region_southwest 1.529044

   [OK] Minden valtozo VIF < 5. Multikollinearitas nem problema.

   Vegso magyarazo valtozok a VIF szures utan: ['age', 'bmi', 'children', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']


In [10]:
# 5. ALAP OLS LINEARIS REGRESSZIOS MODELL
#    (Baseline OLS Linear Regression Model)
# ============================================================================

print("\n" + "=" * 70)
print("5. ALAP OLS LINEARIS REGRESSZIOS MODELL")
print("=" * 70)

# 5.1 - Modell epites
X_base = sm.add_constant(df_encoded[X_cols_final])
y = df_encoded[target]

ols_base = sm.OLS(y, X_base).fit()

# 5.2 - Teljes summary tabla
print("\n--- Alap OLS Modell Summary ---")
print(ols_base.summary())

# Mentjuk a mutatokat az osszehasonlitashoz
base_r2 = ols_base.rsquared
base_adj_r2 = ols_base.rsquared_adj
base_fstat = ols_base.fvalue
base_aic = ols_base.aic
base_bic = ols_base.bic
base_predictions = ols_base.predict(X_base)
base_rmse = np.sqrt(mean_squared_error(y, base_predictions))
base_mae = mean_absolute_error(y, base_predictions)

print(f"\n[STAT] Alapmodell mutatok:")
print(f"   R2           = {base_r2:.6f}")
print(f"   Adj. R2      = {base_adj_r2:.6f}")
print(f"   F-statisztika= {base_fstat:.2f}")
print(f"   RMSE         = {base_rmse:.2f}")
print(f"   MAE          = {base_mae:.2f}")


5. ALAP OLS LINEARIS REGRESSZIOS MODELL

--- Alap OLS Modell Summary ---
                            OLS Regression Results                            
Dep. Variable:                charges   R-squared:                       0.751
Model:                            OLS   Adj. R-squared:                  0.749
Method:                 Least Squares   F-statistic:                     500.0
Date:                Fri, 16 Oct 2026   Prob (F-statistic):               0.00
Time:                        15:25:49   Log-Likelihood:                -13538.
No. Observations:                1337   AIC:                         2.709e+04
Df Residuals:                    1328   BIC:                         2.714e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------

In [11]:
# 6. BACKWARD ELIMINATION (HATRAFELE TORTENO ELIMINACIO)
#    (Backward Stepwise Variable Selection)
# ============================================================================

print("\n" + "=" * 70)
print("6. BACKWARD ELIMINATION (VALTOZOSZELEKC IO)")
print("=" * 70)

alpha = 0.05  # Szignifikanciaszint
print(f"\n   Szignifikanciaszint (alpha) = {alpha}")

# Kezdo valtozok (a VIF szures utani keszletbol)
be_cols = X_cols_final.copy()
step = 0

print(f"\n   Kiindulasi valtozok ({len(be_cols)} db): {be_cols}")
print("-" * 70)

while True:
    step += 1
    X_be = sm.add_constant(df_encoded[be_cols])
    model_be = sm.OLS(y, X_be).fit()

    # p-ertekek lekerdezese (a const-ot kihagyjuk a vizsgalatbol!)
    pvalues = model_be.pvalues.drop("const")

    max_pval = pvalues.max()
    max_pval_var = pvalues.idxmax()

    print(f"\n   [STEP] Lepes #{step}")
    print(f"      Valtozok szama      : {len(be_cols)}")
    print(f"      Adj. R2             : {model_be.rsquared_adj:.6f}")
    print(f"      Legmagasabb p-ertek : {max_pval_var} -> p = {max_pval:.6f}")

    if max_pval > alpha:
        print(f"      [X] '{max_pval_var}' p = {max_pval:.6f} > {alpha} -> KISZURES!")
        be_cols.remove(max_pval_var)
        print(f"      Marado valtozok: {be_cols}")
    else:
        print(f"      [OK] Minden valtozo szignifikans (p <= {alpha}). Megallas.")
        break

# Vegso modell a Backward Elimination utan
print("\n" + "-" * 70)
print("   BACKWARD ELIMINATION - VEGSO MODELL")
print("-" * 70)

X_final = sm.add_constant(df_encoded[be_cols])
ols_final = sm.OLS(y, X_final).fit()
print(ols_final.summary())


6. BACKWARD ELIMINATION (VALTOZOSZELEKC IO)

   Szignifikanciaszint (alpha) = 0.05

   Kiindulasi valtozok (8 db): ['age', 'bmi', 'children', 'sex_male', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']
----------------------------------------------------------------------

   [STEP] Lepes #1
      Valtozok szama      : 8
      Adj. R2             : 0.749247
      Legmagasabb p-ertek : sex_male -> p = 0.697630
      [X] 'sex_male' p = 0.697630 > 0.05 -> KISZURES!
      Marado valtozok: ['age', 'bmi', 'children', 'smoker_yes', 'region_northwest', 'region_southeast', 'region_southwest']

   [STEP] Lepes #2
      Valtozok szama      : 7
      Adj. R2             : 0.749407
      Legmagasabb p-ertek : region_northwest -> p = 0.465152
      [X] 'region_northwest' p = 0.465152 > 0.05 -> KISZURES!
      Marado valtozok: ['age', 'bmi', 'children', 'smoker_yes', 'region_southeast', 'region_southwest']

   [STEP] Lepes #3
      Valtozok szama      : 6
      Adj. R2     

In [12]:
# 7. VEGSO MODELL ERTEKELESE
#    (Final Model Evaluation)
# ============================================================================

print("\n" + "=" * 70)
print("7. VEGSO MODELL ERTEKELESE")
print("=" * 70)

# 7.1 - Vegso modell mutatok
final_r2 = ols_final.rsquared
final_adj_r2 = ols_final.rsquared_adj
final_fstat = ols_final.fvalue
final_aic = ols_final.aic
final_bic = ols_final.bic
final_predictions = ols_final.predict(X_final)
final_rmse = np.sqrt(mean_squared_error(y, final_predictions))
final_mae = mean_absolute_error(y, final_predictions)

print(f"\n[STAT] Vegso modell mutatok:")
print(f"   R2           = {final_r2:.6f}")
print(f"   Adj. R2      = {final_adj_r2:.6f}")
print(f"   F-statisztika= {final_fstat:.2f}")
print(f"   RMSE         = {final_rmse:.2f}")
print(f"   MAE          = {final_mae:.2f}")

# 7.2 - Osszehasonlito tablazat: Alapmodell vs. Vegso modell
print("\n--- Modell-osszehasonlitas ---")
comparison = pd.DataFrame({
    "Mutato": ["R2", "Adj. R2", "F-statisztika", "AIC", "BIC", "RMSE", "MAE",
               "Valtozok szama"],
    "Alapmodell (5. pont)": [
        f"{base_r2:.6f}", f"{base_adj_r2:.6f}", f"{base_fstat:.2f}",
        f"{base_aic:.2f}", f"{base_bic:.2f}", f"{base_rmse:.2f}",
        f"{base_mae:.2f}", f"{len(X_cols_final)}"
    ],
    "Vegso modell (BE, 6. pont)": [
        f"{final_r2:.6f}", f"{final_adj_r2:.6f}", f"{final_fstat:.2f}",
        f"{final_aic:.2f}", f"{final_bic:.2f}", f"{final_rmse:.2f}",
        f"{final_mae:.2f}", f"{len(be_cols)}"
    ]
})
print(comparison.to_string(index=False))


7. VEGSO MODELL ERTEKELESE

[STAT] Vegso modell mutatok:
   R2           = 0.749527
   Adj. R2      = 0.748775
   F-statisztika= 996.48
   RMSE         = 6058.64
   MAE          = 4181.34

--- Modell-osszehasonlitas ---
        Mutato Alapmodell (5. pont) Vegso modell (BE, 6. pont)
            R2             0.750748                   0.749527
       Adj. R2             0.749247                   0.748775
 F-statisztika               499.99                     996.48
           AIC             27094.22                   27092.75
           BIC             27141.00                   27118.74
          RMSE              6043.85                    6058.64
           MAE              4173.49                    4181.34
Valtozok szama                    8                          4


In [13]:
# 8. OLS DIAGNOSZTIKAI VIZSGALATOK A VEGSO MODELLEN
#    (OLS Diagnostic Tests on the Final Model)
# ============================================================================

print("\n" + "=" * 70)
print("8. OLS DIAGNOSZTIKAI VIZSGALATOK")
print("=" * 70)

# Rezidualosok (maradektagok) kiszamitasa
residuals = ols_final.resid
fitted_values = ols_final.fittedvalues
n = len(y)

# ----------------------------------------------------------------------
# 8.1 - HIBATAGOK NORMALITASA (Normality of Residuals)
# ----------------------------------------------------------------------
print("\n" + "-" * 70)
print("8.1 HIBATAGOK NORMALITASA")
print("-" * 70)

# 8.1a - Rezidualosok eloszlasa: Hisztogram + KDE
fig_norm, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(residuals, kde=True, bins=40, color="steelblue", ax=axes[0],
             edgecolor="white", linewidth=0.5, stat="density")
axes[0].set_title("Rezidualosok eloszlasa (Hisztogram + KDE)")
axes[0].set_xlabel("Rezidualosok")
axes[0].set_ylabel("Suruseg")

# 8.1b - Normal Q-Q plot
sm.qqplot(residuals, line="45", fit=True, ax=axes[1],
          markerfacecolor="steelblue", markeredgecolor="white",
          markersize=4, alpha=0.6)
axes[1].set_title("Normal Q-Q Plot")
axes[1].get_lines()[0].set_color("steelblue")
axes[1].get_lines()[1].set_color("red")

plt.tight_layout()
save_and_show(fig_norm, "normality_check")

# 8.1c - Shapiro-Wilk-teszt
# Megjegyzes: Shapiro-Wilk max 5000 mintara mukodik megbizhatoan
if n <= 5000:
    sw_stat, sw_pval = shapiro(residuals)
    print(f"\n   Shapiro-Wilk-teszt:")
    print(f"     Statisztika = {sw_stat:.6f}")
    print(f"     p-ertek     = {sw_pval:.6f}")
    if sw_pval > 0.05:
        print("     -> H0 nem elutasithato: a rezidualosok NORMALIS eloszlasuak (p > 0.05).")
    else:
        print("     -> H0 elutasitva: a rezidualosok NEM normalis eloszlasuak (p <= 0.05).")
else:
    print("   [!] Shapiro-Wilk-teszt kihagyva (n > 5000, nem megbizhato).")

# 8.1d - Jarque-Bera-teszt (nagy mintara is alkalmas)
jb_stat, jb_pval = jarque_bera(residuals)
print(f"\n   Jarque-Bera-teszt:")
print(f"     Statisztika = {jb_stat:.6f}")
print(f"     p-ertek     = {jb_pval:.6f}")
if jb_pval > 0.05:
    print("     -> H0 nem elutasithato: a rezidualosok NORMALIS eloszlasuak (p > 0.05).")
else:
    print("     -> H0 elutasitva: a rezidualosok NEM normalis eloszlasuak (p <= 0.05).")

# ----------------------------------------------------------------------
# 8.2 - HOMOSZKEDASZTICITAS VIZSGALATA (Homoscedasticity)
# ----------------------------------------------------------------------
print("\n" + "-" * 70)
print("8.2 HOMOSZKEDASZTICITAS VIZSGALATA")
print("-" * 70)

# 8.2a - Breusch-Pagan-teszt
bp_stat, bp_pval, bp_fstat, bp_fpval = het_breuschpagan(residuals, X_final)
print(f"\n   Breusch-Pagan-teszt:")
print(f"     LM-statisztika = {bp_stat:.6f}")
print(f"     LM p-ertek     = {bp_pval:.6f}")
print(f"     F-statisztika  = {bp_fstat:.6f}")
print(f"     F p-ertek      = {bp_fpval:.6f}")

if bp_pval > 0.05:
    print("     -> H0 nem elutasithato: HOMOSZKEDASZTICITAS all fenn (p > 0.05).")
    heteroscedasticity_detected = False
else:
    print("     -> H0 elutasitva: HETEROSZKEDASZTICITAS all fenn (p <= 0.05)!")
    heteroscedasticity_detected = True

# 8.2b - Residuals vs. Fitted Values szorodasi abra (+ LOESS trendvonal)
fig_resid, ax = plt.subplots(figsize=(12, 6))

ax.scatter(fitted_values, residuals, alpha=0.4, s=20, color="steelblue",
           edgecolor="white", linewidth=0.3, label="Rezidualosok")
ax.axhline(y=0, color="red", linewidth=1.5, linestyle="--", label="y = 0 referenciavonal")

# LOESS trendvonal (lowess)
try:
    from statsmodels.nonparametric.smoothers_lowess import lowess
    loess_result = lowess(residuals, fitted_values, frac=0.3)
    ax.plot(loess_result[:, 0], loess_result[:, 1], color="darkorange",
            linewidth=2.5, label="LOESS trendvonal")
except ImportError:
    print("   [INFO] LOESS nem elerheto, trendvonal kihagyva.")

ax.set_title("Rezidualosok vs. Becsult ertekek (Residuals vs. Fitted)")
ax.set_xlabel("Becsult ertekek (Fitted Values)")
ax.set_ylabel("Rezidualosok (Residuals)")
ax.legend()
plt.tight_layout()
save_and_show(fig_resid, "residuals_vs_fitted")

# ----------------------------------------------------------------------
# 8.3 - AUTOKORRELACIOS VIZSGALAT (Autocorrelation - Durbin-Watson)
# ----------------------------------------------------------------------
print("\n" + "-" * 70)
print("8.3 AUTOKORRELACIO VIZSGALATA (Durbin-Watson)")
print("-" * 70)

dw_stat = durbin_watson(residuals)
print(f"\n   Durbin-Watson-statisztika: {dw_stat:.4f}")
print(f"   Idealis tartomany: 1.5 - 2.5")

if 1.5 <= dw_stat <= 2.5:
    print("   -> [OK] Nincs szignifikans autokorrelacio a rezidualosokban.")
elif dw_stat < 1.5:
    print("   -> [!] Pozitiv autokorrelacio lehetseges (DW < 1.5)!")
else:
    print("   -> [!] Negativ autokorrelacio lehetseges (DW > 2.5)!")

# ----------------------------------------------------------------------
# 8.4 - KIUGRO ES BEFOLYASOS PONTOK (Cook's Distance)
# ----------------------------------------------------------------------
print("\n" + "-" * 70)
print("8.4 KIUGRO ES BEFOLYASOS PONTOK (Cook's Distance)")
print("-" * 70)

# Cook-tavolsag kiszamitasa
influence = ols_final.get_influence()
cooks_d = influence.cooks_distance[0]

# Kritikus kuszobErtek: 4/n
cooks_threshold = 4 / n
print(f"\n   Kritikus kuszobErtek (4/n) : {cooks_threshold:.6f}")
print(f"   Megfigyelesek szama (n)   : {n}")

# Befolyasos pontok azonositasa
influential_idx = np.where(cooks_d > cooks_threshold)[0]
print(f"   Befolyasos pontok szama   : {len(influential_idx)} "
      f"({len(influential_idx)/n*100:.1f}%)")

# Cook-tavolsagok diagram
fig_cook, ax = plt.subplots(figsize=(14, 6))
ax.stem(range(n), cooks_d, linefmt="steelblue", markerfmt=",", basefmt=" ")
ax.axhline(y=cooks_threshold, color="red", linewidth=1.5, linestyle="--",
           label=f"Kuszob (4/n = {cooks_threshold:.4f})")
ax.set_title("Cook-tavolsagok (Cook's Distance)", fontsize=14, fontweight="bold")
ax.set_xlabel("Megfigyeles indexe")
ax.set_ylabel("Cook-tavolsag")
ax.legend()
plt.tight_layout()
save_and_show(fig_cook, "cooks_distance")

# A legbefolyasosabb 10 pont kilistazasa
if len(influential_idx) > 0:
    top_n = min(10, len(influential_idx))
    top_influential = np.argsort(cooks_d)[-top_n:][::-1]
    print(f"\n   Top {top_n} legbefolyasosabb megfigyeles:")
    print(f"   {'Index':>7} | {'Cook-tavolsag':>15}")
    print(f"   {'-'*7}-+-{'-'*15}")
    for idx in top_influential:
        print(f"   {idx:>7} | {cooks_d[idx]:>15.6f}")
else:
    print("\n   [OK] Nincsenek befolyasos pontok a 4/n kuszob felett.")

# ----------------------------------------------------------------------
# 8.5 - ROBUSZTUS STANDARD HIBAK (HC3) - ha heteroszkedaszticitas van
# ----------------------------------------------------------------------
if heteroscedasticity_detected:
    print("\n" + "-" * 70)
    print("8.5 ROBUSZTUS STANDARD HIBAK (HC3 korrekcio)")
    print("-" * 70)
    print("\n   [!] A Breusch-Pagan-teszt heteroszkedaszticitas-t jelzett.")
    print("   -> Robusztus standard hibakkal korrigalt modell (cov_type='HC3'):\n")

    ols_robust = sm.OLS(y, X_final).fit(cov_type="HC3")
    print(ols_robust.summary())

    print("\n   [INFO] Osszehasonlitas: Hagyomanyos vs. HC3 robusztus standard hibak")
    se_comparison = pd.DataFrame({
        "Valtozo": ols_final.params.index,
        "Koefficiens": ols_final.params.values.round(4),
        "SE (Hagyomanyos)": ols_final.bse.values.round(4),
        "SE (HC3 Robusztus)": ols_robust.bse.values.round(4),
        "p (Hagyomanyos)": ols_final.pvalues.values.round(6),
        "p (HC3 Robusztus)": ols_robust.pvalues.values.round(6),
    })
    print(se_comparison.to_string(index=False))
else:
    print("\n" + "-" * 70)
    print("8.5 ROBUSZTUS STANDARD HIBAK")
    print("-" * 70)
    print("\n   [OK] A Breusch-Pagan-teszt nem jelzett heteroszkedaszticitas-t.")
    print("   -> HC3 korrekcio nem szukseges.")


8. OLS DIAGNOSZTIKAI VIZSGALATOK

----------------------------------------------------------------------
8.1 HIBATAGOK NORMALITASA
----------------------------------------------------------------------
   [ABRA MENTVE] -> c:\Users\hp\Antigravity\plots\09_normality_check.png

   Shapiro-Wilk-teszt:
     Statisztika = 0.899716
     p-ertek     = 0.000000
     -> H0 elutasitva: a rezidualosok NEM normalis eloszlasuak (p <= 0.05).

   Jarque-Bera-teszt:
     Statisztika = 719.880154
     p-ertek     = 0.000000
     -> H0 elutasitva: a rezidualosok NEM normalis eloszlasuak (p <= 0.05).

----------------------------------------------------------------------
8.2 HOMOSZKEDASZTICITAS VIZSGALATA
----------------------------------------------------------------------

   Breusch-Pagan-teszt:
     LM-statisztika = 116.979913
     LM p-ertek     = 0.000000
     F-statisztika  = 31.929238
     F p-ertek      = 0.000000
     -> H0 elutasitva: HETEROSZKEDASZTICITAS all fenn (p <= 0.05)!
   [ABRA MENTV

In [14]:
# ZARO OSSZEFOGLALO (Final Summary)
# ============================================================================
print("\n" + "=" * 70)
print("ZARO OSSZEFOGLALO")
print("=" * 70)
print(f"""
[INFO] Vegso modell valtozoi: {be_cols}
[STAT] R2     = {final_r2:.4f}  ({final_r2*100:.2f}%)
[STAT] Adj R2 = {final_adj_r2:.4f}  ({final_adj_r2*100:.2f}%)
[STAT] RMSE   = {final_rmse:.2f}
[STAT] MAE    = {final_mae:.2f}

[INFO] A modell a celvaltozo (charges) variancipjpnak ~{final_r2*100:.1f}%-at magyarazza.

Megjegyzes:
  * Ha a normalitasi tesztek elutasitjak H0-t, erdemes lehet
    log-transzformaciot alkalmazni a celvaltozora.
  * Ha heteroszkedaszticitas all fenn, a HC3 robusztus standard hibak
    megbizhatobb kovetkezteteseket biztositanak.
  * A Cook-tavolsag alapjan azonositott befolyasos pontok tovabbi
    vizsgalatot igenyelhetnek (pl. domain-specifikus elemzes).
""")

print("[OK] Az elemzes befejezodott.")
print(f"[INFO] Minden abra itt talalhato: {OUTPUT_DIR}")


ZARO OSSZEFOGLALO

[INFO] Vegso modell valtozoi: ['age', 'bmi', 'children', 'smoker_yes']
[STAT] R2     = 0.7495  (74.95%)
[STAT] Adj R2 = 0.7488  (74.88%)
[STAT] RMSE   = 6058.64
[STAT] MAE    = 4181.34

[INFO] A modell a celvaltozo (charges) variancipjpnak ~75.0%-at magyarazza.

Megjegyzes:
  * Ha a normalitasi tesztek elutasitjak H0-t, erdemes lehet
    log-transzformaciot alkalmazni a celvaltozora.
  * Ha heteroszkedaszticitas all fenn, a HC3 robusztus standard hibak
    megbizhatobb kovetkezteteseket biztositanak.
  * A Cook-tavolsag alapjan azonositott befolyasos pontok tovabbi
    vizsgalatot igenyelhetnek (pl. domain-specifikus elemzes).

[OK] Az elemzes befejezodott.
[INFO] Minden abra itt talalhato: c:\Users\hp\Antigravity\plots
